# VIPER — Deepfake Detection Demo
**Video Identity Perturbation and Extraction Residual**

Robin Singh · Bennett University · 2025

This notebook runs the full VIPER pipeline:
1. Install dependencies
2. Clone repo
3. Download a test video
4. Run detection (analytical mode — no checkpoint needed)
5. Launch Gradio UI

**Runtime:** T4 GPU recommended. CPU works for analytical mode.

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────
!pip install -q torch torchvision
!pip install -q insightface onnxruntime-gpu
!pip install -q mediapipe
!pip install -q opencv-python-headless scipy scikit-learn
!pip install -q gradio matplotlib tqdm
print('✓ Dependencies installed')

In [ ]:
# ── Step 2: Clone VIPER repo ───────────────────────────────────
import os
if not os.path.exists('VIPER'):
    !git clone https://github.com/rxbinsingh/VIPER
%cd VIPER
print('✓ Repository ready')

In [ ]:
# ── Step 3: Quick test with a sample video ─────────────────────
# Download a short test video (replace with your own video path)
# Option A: Upload your own video
from google.colab import files
print('Upload a video file (mp4, avi, mov):')
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f'✓ Uploaded: {video_path}')

In [ ]:
# ── Step 4: Run VIPER detection ────────────────────────────────
import sys
sys.path.insert(0, '.')

from src.viper_complete import VIPERDetector

# Analytical mode (no checkpoint needed)
detector = VIPERDetector(checkpoint=None)

print(f'Analyzing: {video_path}')
result = detector.detect(video_path)

print('\n' + '='*50)
print(f'PREDICTION:  {result["prediction"]}')
print(f'CONFIDENCE:  {result["confidence"]*100:.1f}%')
print(f'VIPER SCORE: {result["viper_score"]:.4f}  (>0.5 = FAKE)')
print('='*50)
print('\nSignal Breakdown:')
for name, sig in result['signals'].items():
    if sig['score'] is not None:
        status = '⚠️  TRIGGERED' if sig['triggered'] else '✅  OK'
        print(f'  {name:20s}: {sig["score"]:.4f}  {status}')
print(f'\nFrames analyzed: {result["frames_analyzed"]}')
print(f'Anchor quality:  {result["anchor_quality"]:.3f}')

In [ ]:
# ── Step 5: Plot reaction curve ────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
color = '#d62728' if result['prediction'] == 'FAKE' else '#2ca02c'

for ax, seq, title, threshold in zip(
    axes,
    [result['gir_sequence'], result['tfr_sequence']],
    ['Geometric Identity Residual (GIR)', 'Texture Frequency Residual (TFR)'],
    [0.35, 0.08],
):
    frames = list(range(len(seq)))
    ax.plot(frames, seq, color=color, linewidth=2.5, label='Residual r(t)')
    ax.axhline(threshold, color='gray', linestyle='--',
               linewidth=1.5, label=f'Threshold ({threshold})')
    ax.fill_between(frames, seq, threshold,
                    where=[s > threshold for s in seq],
                    alpha=0.3, color=color, label='Violation zone')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Frame index', fontsize=10)
    ax.set_ylabel('Residual energy', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f'VIPER Reaction Curve  |  Prediction: {result["prediction"]}  |  Score: {result["viper_score"]:.4f}',
    fontsize=12, fontweight='bold', color=color,
)
plt.tight_layout()
plt.savefig('reaction_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved reaction_curve.png')

In [ ]:
# ── Step 6: Launch Gradio UI ───────────────────────────────────
# Opens a public URL you can share with your internship supervisor
!python app.py

## Training on CelebDF-v2

To train the full model (improves AUC from ~82% to ~91%):

```python
# 1. Download CelebDF-v2 (free, no form)
# https://github.com/yuezunli/celeb-deepfakeforensics

# 2. Train (~2.5 hours on T4)
!python train.py \
    --data_dir /path/to/celeb-df-v2 \
    --epochs 15 \
    --batch_size 16

# 3. Evaluate
!python eval/evaluate.py \
    --data_dir /path/to/celeb-df-v2 \
    --checkpoint checkpoints/viper_best.pt

# 4. Run ablation
!python eval/ablation.py --data_dir /path/to/celeb-df-v2
```